In [1]:
import pandas as pd
import networkx as nx
from itertools import combinations

In [3]:
days = [1, 30, 60, 90]

graphs = {}
attrs = {}

for d in days:
    edges = pd.read_csv(f"Part_B/connections_day_{d}.csv")
    nodes = pd.read_csv(f"Part_B/properties_day_{d}.csv")
    
    G = nx.Graph()
    G.add_edges_from(zip(edges['node_i'], edges['node_j']))
    
    nx.set_node_attributes(G, nodes.set_index('id').to_dict('index'))
    
    graphs[d] = G
    attrs[d] = nodes

In [ ]:
def triadic_closures(G_prev, G_next):
    closures = []
    for u in G_prev.nodes():
        neighbors = list(G_prev.neighbors(u))
        for v, w in combinations(neighbors, 2):
            if not G_prev.has_edge(v, w) and G_next.has_edge(v, w):
                closures.append((v, w))
    return closures


triadic_events = {}
for d1, d2 in zip(days[:-1], days[1:]):
    triadic_events[(d1, d2)] = triadic_closures(graphs[d1], graphs[d2])


2935


In [27]:
def smoker_triadic_events(G_prev, events):
    smokers = {n for n,d in G_prev.nodes(data=True) if d['smokes']==1}
    return [e for e in events if e[0] in smokers or e[1] in smokers]


for d1, d2 in zip(days[:-1], days[1:]):
    all_events = triadic_events[(d1, d2)]
    smoker_events = smoker_triadic_events(graphs[d1], all_events)
    ratio = len(smoker_events) / len(all_events)
    print(f"the ratio from {d1} to {d2} based on triadic closure is {ratio}")




the ratio from 1 to 30 based on triadic closure is 0.3333333333333333
the ratio from 30 to 60 based on triadic closure is 0.9297200714711138
the ratio from 60 to 90 based on triadic closure is 0.787052810902896


In [24]:
def smoker_membership_ratio(G):
    same, total = 0, 0
    for u, v in G.edges():
        if G.nodes[u]['smokes'] == 1 or G.nodes[v]['smokes'] == 1:
            total += 1
            if G.nodes[u]['class_number'] == G.nodes[v]['class_number']:
                same += 1
    return same / total if total > 0 else 0


def nonsmoker_membership_ratio(G):
    same, total = 0, 0
    for u, v in G.edges():
        if G.nodes[u]['smokes'] == 0 and G.nodes[v]['smokes'] == 0:
            total += 1
            if G.nodes[u]['class_number'] == G.nodes[v]['class_number']:
                same += 1
    return same / total if total > 0 else 0


In [25]:
for d in days:
    print(
        d,
        smoker_membership_ratio(graphs[d]),
        nonsmoker_membership_ratio(graphs[d])
    )

1 0.2 0.15
30 0.5565217391304348 0.7560975609756098
60 0.3765932792584009 0.6918238993710691
90 0.7230769230769231 0.8909090909090909
